In [1]:
import sys
import asyncio
import nest_asyncio

# Apply nest_asyncio to allow async Playwright inside Jupyter Notebook
nest_asyncio.apply()

# Fix Windows Proactor Event Loop for Jupyter Subprocesses
if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

print("Environment setup complete and Windows Async Policy applied.")

Environment setup complete and Windows Async Policy applied.


In [2]:
import os

# Project Root Directory Setup
PROJECT_ROOT = os.getcwd()
DATA_RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

PATHS = {
    "web_pages": os.path.join(DATA_RAW_DIR, "web_pages"),
    "pdfs": os.path.join(DATA_RAW_DIR, "pdfs"),
    "docs": os.path.join(DATA_RAW_DIR, "docs"),
    "tables": os.path.join(DATA_RAW_DIR, "tables"),
    "metadata": os.path.join(DATA_PROCESSED_DIR, "metadata.json")
}

# Create required directories if they don't exist
for path in PATHS.values():
    if path.endswith(".json"):
        os.makedirs(os.path.dirname(path), exist_ok=True)
    else:
        os.makedirs(path, exist_ok=True)

print("Directory structure created successfully at:", PROJECT_ROOT)

Directory structure created successfully at: E:\Rasengan\College-Knowledge-Search


In [3]:
# Master Target URLs List
URL_LIST = [
    "https://www.iiitn.ac.in/",
    "https://www.iiitn.ac.in/about",
    "https://www.iiitn.ac.in/institute/institutesprofile",
    "https://www.iiitn.ac.in/governance/administration",
    "https://www.iiitn.ac.in/governance/committee",
    "https://www.iiitn.ac.in/pages/campusgallery",
    "https://www.iiitn.ac.in/institute/facilities",
    "https://www.iiitn.ac.in/notices",
    "https://www.iiitn.ac.in/pages/reservation-policy",
    "https://www.iiitn.ac.in/departments",
    "https://www.iiitn.ac.in/departments/basic_science",
    "https://www.iiitn.ac.in/departments/cse",
    "https://www.iiitn.ac.in/departments/ece",
    "https://www.iiitn.ac.in/academics/courses",
    "https://www.iiitn.ac.in/academics/curricula",
    "https://www.iiitn.ac.in/academics/timetable",
    "https://www.iiitn.ac.in/pages/academicfee",
    "https://www.iiitn.ac.in/admissions/btech",
    "https://www.iiitn.ac.in/admissions/btech/acadfees",
    "https://www.iiitn.ac.in/admissions/btech/hostelfees",
    "https://www.iiitn.ac.in/pages/scholarships",
    "https://www.iiitn.ac.in/pages/formats",
    "https://www.iiitn.ac.in/pages/loanschemes",
    "https://www.iiitn.ac.in/admissions/pg",
    "https://www.iiitn.ac.in/phd/overview",
    "https://www.iiitn.ac.in/phd/resources",
    "https://www.iiitn.ac.in/phd/operational-details",
    "https://www.iiitn.ac.in/phd/structure",
    "https://www.iiitn.ac.in/research/publications",
    "https://www.iiitn.ac.in/research/patents",
    "https://www.iiitn.ac.in/research/projects",
    "https://www.iiitn.ac.in/research/consultancy",
    "https://www.iiitn.ac.in/placements",
    "https://www.iiitn.ac.in/placements/internships",
    "https://www.iiitn.ac.in/placements/statistics",
    "https://www.iiitn.ac.in/placements/students",
    "https://www.iiitn.ac.in/placements/companies",
    "https://www.iiitn.ac.in/placements/contact",
    "https://www.iiitn.ac.in/pages/nirf",
    "https://www.iiitn.ac.in/pages/rti",
    "https://www.iiitn.ac.in/institute/annualreports",
    "https://www.iiitn.ac.in/institute/annualaccounts",
    "https://www.iiitn.ac.in/institute/rtireports",
    "https://www.iiitn.ac.in/institute/rtidetails",
    "https://www.iiitn.ac.in/institute/rticontact",
    "https://www.iiitn.ac.in/institute/minutesofmeeting",
    "https://www.iiitn.ac.in/institute/immutableproperties",
    "https://www.iiitn.ac.in/pages/tenders",
    "https://www.iiitn.ac.in/pages/iic",
    "https://www.iiitn.ac.in/pages/elclub",
    "https://www.iiitn.ac.in/pages/guesthouse",
    "https://www.iiitn.ac.in/pages/pressrelease",
    "https://www.iiitn.ac.in/pages/officialdocuments",
    "https://www.iiitn.ac.in/academics/calendar",
    "https://www.iiitn.ac.in/academics/library",
    "https://www.iiitn.ac.in/founders",
    "https://www.iiitn.ac.in/students",
    "https://www.iiitn.ac.in/students/hostellife",
    "https://www.iiitn.ac.in/pages/studentclubs",
    "https://www.iiitn.ac.in/events",
    "https://www.iiitn.ac.in/pages/studentachievements",
    "https://www.iiitn.ac.in/pages/mess",
    "https://www.iiitn.ac.in/pages/clinicalcounselling",
    "https://www.iiitn.ac.in/facultyandstaff",
    "https://www.iiitn.ac.in/pages/directory",
    "https://www.iiitn.ac.in/pages/facultyachievements",
    "https://www.iiitn.ac.in/alumni",
    "https://www.iiitn.ac.in/recruitments",
    "https://www.iiitn.ac.in/visitors",
    "https://www.iiitn.ac.in/pages/eoc",
    "https://www.iiitn.ac.in/pages/icc",
    "https://www.iiitn.ac.in/pages/antiragging",
    "https://www.iiitn.ac.in/pages/grievance_committee_sc_st",
    "https://www.iiitn.ac.in/pages/grievance"
]

In [4]:
import re
import hashlib
from urllib.parse import urlparse

HTTP_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

metadata_catalog = []
downloaded_hashes = set()

def sanitize_filename(name: str) -> str:
    return re.sub(r'[^a-zA-Z0-9_\-\.]', '_', name)

def compute_hash(data: bytes) -> str:
    return hashlib.md5(data).hexdigest()

def classify_url(url: str) -> str:
    path = urlparse(url).path.lower()
    if "/placements" in path:
        return "Placements"
    elif "/academics" in path or "/departments" in path or "/curricula" in path or "/courses" in path or "/timetable" in path:
        return "Academics"
    elif "/admissions" in path or "/phd" in path:
        return "Admissions"
    elif "/governance" in path or "/committee" in path:
        return "Governance"
    elif "/notices" in path or "/tenders" in path or "/officialdocuments" in path:
        return "Notices"
    elif "/research" in path:
        return "Research"
    elif "/students" in path or "/hostellife" in path or "/mess" in path:
        return "StudentAffairs"
    elif "/institute" in path or "/about" in path or "/facilities" in path:
        return "Institute"
    return "General"

def convert_google_sheets_url(url: str) -> str:
    if "docs.google.com/spreadsheets" in url:
        base = re.sub(r'/(pubhtml|edit).*$', '', url)
        gid_match = re.search(r'gid=(\d+)', url)
        gid_str = f"&gid={gid_match.group(1)}" if gid_match else ""
        return f"{base}/pub?output=csv{gid_str}"
    return url

def is_target_file_link(href: str, link_text: str) -> bool:
    """Detects PDFs, Word docs, Excel files, Google Sheets, and schedules via URL or link text."""
    href_lower = href.lower()
    text_lower = link_text.lower()

    # File Extension Check
    valid_extensions = ('.pdf', '.docx', '.doc', '.xlsx', '.xls', '.csv')
    if any(href_lower.endswith(ext) or f"{ext}?" in href_lower for ext in valid_extensions):
        return True

    # Google Sheets Check
    if "docs.google.com/spreadsheets" in href_lower:
        return True

    # Schedule / Word Doc Keyword Fallback
    schedule_keywords = ('schedule', 'timetable', 'class', 'lab', 'time-table', 'docx', 'doc')
    if any(kw in text_lower or kw in href_lower for kw in schedule_keywords):
        if not href_lower.startswith(('javascript:', 'mailto:', '#')):
            return True

    return False

print("Updated utilities loaded with enhanced DOCX & Schedule detection.")

Updated utilities loaded with enhanced DOCX & Schedule detection.


In [5]:
import os
import requests
import io
from PIL import Image

try:
    import pytesseract
    OCR_AVAILABLE = True
except ImportError:
    OCR_AVAILABLE = False

def stream_download_file(url: str, category: str, parent_page: str, link_text: str) -> dict | None:
    try:
        download_url = convert_google_sheets_url(url)
        response = requests.get(download_url, headers=HTTP_HEADERS, stream=True, timeout=60)
        if response.status_code != 200:
            return None

        file_bytes = bytearray()
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                file_bytes.extend(chunk)

        file_hash = compute_hash(bytes(file_bytes))
        if file_hash in downloaded_hashes:
            return None
        downloaded_hashes.add(file_hash)

        parsed_url = urlparse(url)
        ext = os.path.splitext(parsed_url.path)[1].lower()

        if "docs.google.com/spreadsheets" in url:
            ext = ".csv"
        elif not ext or ext not in ['.pdf', '.docx', '.doc', '.xlsx', '.xls', '.csv']:
            # Default fallback based on link text
            if 'doc' in link_text.lower():
                ext = '.docx'
            elif 'pdf' in link_text.lower():
                ext = '.pdf'
            else:
                ext = '.docx'

        if ext == ".pdf":
            target_dir = os.path.join(PATHS["pdfs"], category)
            file_type = "pdf"
        elif ext in [".docx", ".doc"]:
            target_dir = os.path.join(PATHS["docs"], category)
            file_type = "docx"
        elif ext in [".xlsx", ".xls", ".csv"]:
            target_dir = os.path.join(PATHS["tables"], category)
            file_type = "table"
        else:
            return None

        os.makedirs(target_dir, exist_ok=True)
        filename = sanitize_filename(os.path.basename(parsed_url.path) or f"doc_{file_hash[:8]}{ext}")
        if not filename.endswith(ext):
            filename += ext

        full_path = os.path.join(target_dir, filename)
        with open(full_path, "wb") as f:
            f.write(file_bytes)

        rel_path = os.path.relpath(full_path, PROJECT_ROOT).replace("\\", "/")

        return {
            "doc_id": f"{file_type}_{file_hash[:10]}",
            "title": link_text if link_text else filename,
            "category": category,
            "file_type": file_type,
            "source_url": url,
            "parent_page_url": parent_page,
            "local_path": rel_path,
            "content_hash": file_hash
        }

    except Exception as e:
        print(f"   [ERROR Streaming File] {url}: {e}")
        return None

def extract_and_ocr_images(soup, page_url: str, category: str) -> str:
    """Extracts placement/recruiter company images and extracts text using Tesseract OCR."""
    ocr_results = []
    images = soup.find_all("img", src=True)

    if not images:
        return ""

    img_dir = os.path.join(PATHS.get("images", os.path.join(DATA_RAW_DIR, "images")), category)
    os.makedirs(img_dir, exist_ok=True)

    for idx, img in enumerate(images, 1):
        src = img["src"].strip()
        img_url = urljoin(page_url, src)
        alt_text = img.get("alt", "").strip()

        # Skip small icons/logos
        if "logo" in src.lower() and "iiitn" in src.lower():
            continue

        try:
            res = requests.get(img_url, headers=HTTP_HEADERS, timeout=15)
            if res.status_code == 200 and len(res.content) > 3000: # Ignore tiny graphics
                img_hash = compute_hash(res.content)
                img_filename = sanitize_filename(f"placement_img_{img_hash[:8]}.png")
                img_path = os.path.join(img_dir, img_filename)

                with open(img_path, "wb") as f:
                    f.write(res.content)

                text_content = f"Image Alt Text: {alt_text}\n"

                if OCR_AVAILABLE:
                    try:
                        image = Image.open(io.BytesIO(res.content))
                        ocr_text = pytesseract.image_to_string(image).strip()
                        if ocr_text:
                            text_content += f"OCR Extracted Companies/Text:\n{ocr_text}\n"
                    except Exception:
                        pass

                ocr_results.append(text_content)
        except Exception:
            pass

    if ocr_results:
        return "\n\n=== RECRUITER & PLACEMENT IMAGE OCR DATA ===\n" + "\n".join(ocr_results)
    return ""

print("Enhanced downloader and Placement Image OCR module initialized.")

Enhanced downloader and Placement Image OCR module initialized.


In [6]:
import asyncio
import json
from urllib.parse import urljoin
from bs4 import BeautifulSoup
from playwright.sync_api import sync_playwright

def _playwright_sync_worker():
    print("=== STARTING PLAYWRIGHT DATA COLLECTION ENGINE ===")
    unique_urls = list(dict.fromkeys(URL_LIST))
    print(f"Total Target Pages: {len(unique_urls)}")

    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        context = browser.new_context(user_agent=HTTP_HEADERS["User-Agent"])
        page = context.new_page()

        for idx, target_url in enumerate(unique_urls, 1):
            category = classify_url(target_url)
            print(f"\n[{idx}/{len(unique_urls)}] Rendering ({category}) -> {target_url}")

            try:
                response = page.goto(target_url, wait_until="networkidle", timeout=30000)
                if not response or response.status != 200:
                    print(f"   [WARNING] HTTP Status: {response.status if response else 'No Response'}")
                    continue

                html_content = page.content()
                soup = BeautifulSoup(html_content, "html.parser")

                # 1. Extract File Attachments (.pdf, .docx, Class/Lab Schedules, Google Sheets)
                found_links = soup.find_all("a", href=True)
                print(f"   Found {len(found_links)} rendered links in DOM")

                for a_tag in found_links:
                    href = urljoin(target_url, a_tag["href"].strip())
                    link_text = a_tag.get_text(strip=True)

                    if is_target_file_link(href, link_text):
                        record = stream_download_file(href, category, target_url, link_text)
                        if record:
                            metadata_catalog.append(record)
                            print(f"   [SAVED ASSET] -> {record['local_path']}")

                # 2. Extract Placement / Recruiter Images & Run OCR
                ocr_data = ""
                if category == "Placements":
                    ocr_data = extract_and_ocr_images(soup, target_url, category)

                # 3. Extract Webpage Body Text
                for tag in soup(["script", "style", "noscript"]):
                    tag.extract()

                page_title = soup.title.string.strip() if soup.title else target_url
                main_text = soup.get_text(separator="\n", strip=True)

                if ocr_data:
                    main_text += ocr_data

                page_bytes = main_text.encode('utf-8')
                page_hash = compute_hash(page_bytes)

                if page_hash not in downloaded_hashes and len(main_text) > 50:
                    downloaded_hashes.add(page_hash)
                    web_dir = os.path.join(PATHS["web_pages"], category)
                    os.makedirs(web_dir, exist_ok=True)

                    filename = sanitize_filename(f"{page_title[:30]}_{page_hash[:6]}.txt")
                    filepath = os.path.join(web_dir, filename)

                    with open(filepath, "w", encoding="utf-8") as f:
                        f.write(f"Title: {page_title}\nURL: {target_url}\nCategory: {category}\n\n{main_text}")

                    rel_web_path = os.path.relpath(filepath, PROJECT_ROOT).replace("\\", "/")

                    metadata_catalog.append({
                        "doc_id": f"html_{page_hash[:10]}",
                        "title": page_title,
                        "category": category,
                        "file_type": "html",
                        "source_url": target_url,
                        "parent_page_url": target_url,
                        "local_path": rel_web_path,
                        "content_hash": page_hash
                    })
                    print(f"   [SAVED WEBPAGE TEXT] -> {rel_web_path}")

            except Exception as e:
                print(f"   [ERROR Page Processing] {target_url}: {e}")

        browser.close()

    # Save Catalog
    with open(PATHS["metadata"], "w", encoding="utf-8") as f:
        json.dump(metadata_catalog, f, indent=2)

    print("\n=== DATA COLLECTION COMPLETE ===")
    print(f"Total Assets Cataloged: {len(metadata_catalog)}")
    print(f"Metadata catalog saved to: {PATHS['metadata']}")

async def run_collection_in_thread():
    await asyncio.to_thread(_playwright_sync_worker)

print("Updated extraction worker ready.")

Updated extraction worker ready.


In [7]:
await run_collection_in_thread()

=== STARTING PLAYWRIGHT DATA COLLECTION ENGINE ===
Total Target Pages: 74

[1/74] Rendering (General) -> https://www.iiitn.ac.in/
   Found 134 rendered links in DOM
   [SAVED ASSET] -> data/raw/docs/General/timetable.docx
   [SAVED ASSET] -> data/raw/pdfs/General/fb94fbf05b774a7e80dfef4a125c4dcb.pdf
   [SAVED ASSET] -> data/raw/docs/General/viewform.docx
   [SAVED ASSET] -> data/raw/pdfs/General/0bc8b07c132a489db9adc595cd45f98e.pdf
   [SAVED WEBPAGE TEXT] -> data/raw/web_pages/General/Indian_Institute_of_Informatio_3164bb.txt

[2/74] Rendering (Institute) -> https://www.iiitn.ac.in/about
   Found 131 rendered links in DOM
   [SAVED ASSET] -> data/raw/pdfs/Institute/65bb626447f645ec9264204806ff8911.pdf
   [SAVED ASSET] -> data/raw/pdfs/Institute/bd435397ed444448914410ade813746c.pdf
   [SAVED ASSET] -> data/raw/docs/Institute/viewform.docx
   [SAVED WEBPAGE TEXT] -> data/raw/web_pages/Institute/Indian_Institute_of_Informatio_c625a9.txt

[3/74] Rendering (Institute) -> https://www.iiitn.a

In [8]:
import json

if os.path.exists(PATHS["metadata"]):
    with open(PATHS["metadata"], "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"Total Cataloged Documents: {len(data)}\n")

    # Summary by file type
    file_types = {}
    for item in data:
        ft = item.get("file_type", "unknown")
        file_types[ft] = file_types.get(ft, 0) + 1

    print("Asset Distribution:")
    for ft, count in file_types.items():
        print(f"  - {ft.upper()}: {count} files")

    print("\nFirst 3 Sample Entries:")
    print(json.dumps(data[:3], indent=2))
else:
    print("Metadata file not found yet. Run Cell 7 first.")

Total Cataloged Documents: 410

Asset Distribution:
  - DOCX: 68 files
  - PDF: 276 files
  - HTML: 65 files
  - TABLE: 1 files

First 3 Sample Entries:
[
  {
    "doc_id": "docx_11b975a522",
    "title": "Time Table",
    "category": "General",
    "file_type": "docx",
    "source_url": "https://www.iiitn.ac.in/academics/timetable",
    "parent_page_url": "https://www.iiitn.ac.in/",
    "local_path": "data/raw/docs/General/timetable.docx",
    "content_hash": "11b975a5227c8683cbe74ed5533b780e"
  },
  {
    "doc_id": "pdf_d34d247be2",
    "title": "Telephone Directory",
    "category": "General",
    "file_type": "pdf",
    "source_url": "https://iiitn.ac.in/media/fb94fbf05b774a7e80dfef4a125c4dcb.pdf",
    "parent_page_url": "https://www.iiitn.ac.in/",
    "local_path": "data/raw/pdfs/General/fb94fbf05b774a7e80dfef4a125c4dcb.pdf",
    "content_hash": "d34d247be2c420f02ba089b896147fba"
  },
  {
    "doc_id": "docx_fb6f75d84f",
    "title": "Networking Complaint",
    "category": "Genera